In [1]:
import pandas as pd
import torch
import transformers
from torch.utils.data import Dataset
import os
import numpy as np
from torch.utils.data import DataLoader
from torch.optim import lr_scheduler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

/blue/egn6933/share/apatil2/conda/envs/pathogen_new1/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
combined = pd.read_csv('combined_df.csv')

In [3]:
combined.shape

(781867, 5)

In [4]:
type(combined['Pathogenicity'].value_counts())

pandas.Series

In [5]:
pos = (combined['Pathogenicity']==1).sum()
neg = (combined['Pathogenicity']==0).sum()

pos/neg

np.float64(0.06861190761905023)

In [6]:
folder = '/blue/egn6933/apatil2/embeddings/'

files = sorted([f for f in os.listdir(folder) if f.endswith('.pt')])

train_files, test_files = train_test_split(
    files,
    test_size=0.2,
    random_state=42
)

print(len(train_files), len(test_files))

2444 611


In [7]:
# print(type(train), len(train))

In [8]:
# define the dataset class
class EmbeddingDataset(Dataset):
    def __init__(self, folder, files):
        self.folder = folder
        self.files = files
    
    
    def __len__(self):
        
        return len(self.files)
    
    def __getitem__(self, idx):
        batch = torch.load(os.path.join(self.folder, self.files[idx]))
        # print(type(batch), len(batch))
        X = batch['X'][:, 64, :]
        y = batch['Y']
        
        return X, y                        

In [9]:
# define the data loader

train_dataset = EmbeddingDataset(folder, train_files)

test_dataset = EmbeddingDataset(folder, test_files)

train_loader = DataLoader(train_dataset, batch_size = 1, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False)

In [10]:
import torch.nn as nn

class MLP(nn.Module):
    
    def __init__(self):
        super().__init__()
    
        self.model = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, 1)
        )
    
    def forward(self, x):
        x = torch.nn.functional.normalize(x, dim=1)
        return self.model(x)
    

In [11]:
# #training configuration

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = MLP().to(device)

# #using weighted loss function
# weights = torch.sqrt(torch.tensor([neg/pos])).to(device)
# criterion = nn.BCEWithLogitsLoss(pos_weight = weights)

# optimizer = torch.optim.Adam(
#     model.parameters(),
#     lr = 1e-4
# )

# epoch = 15

In [12]:
#complex model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model1 = MLP().to(device)

#using weighted loss function
weights = torch.sqrt(torch.tensor([neg/pos])).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight = weights)

initial_rate = 1e-4
optimizer = torch.optim.Adam(
    model1.parameters(),
    lr = initial_rate
)

scheduler = lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)

epoch = 25

In [13]:
def compute_accuracy(preds, labels):
    preds = torch.sigmoid(preds)
    preds = (preds>0.5).float()
    
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    
    return correct, total

In [14]:
def compute_precision(preds, labels):
    preds = torch.sigmoid(preds)
    preds = (preds>0.5).float()
    
    true_positive = ((preds==1.0) & (labels==1.0)).sum().item()
    false_positive = ((preds==1.0) & (labels==0.0)).sum().item()
    
    return true_positive, false_positive

In [15]:
def compute_recall(preds, labels):
    preds = torch.sigmoid(preds)
    preds = (preds>0.5).float()
    
    false_negative = ((preds==0.0) & (labels==1.0)).sum().item()
    return false_negative

In [ ]:
# training loop
# model.train()
train_loss = []
train_acc = []
train_precision = []
train_recall = []

for i in range(epoch):
    model1.train()
    total_correct = 0
    total_samples = 0
    total_loss = 0
    total_true_positive = 0
    total_false_positive = 0
    total_false_negative = 0
    
    for X,y in train_loader:
        # print(X.shape, y.shape)
        X = X.squeeze().to(device)
        y = y.squeeze().float().to(device)
        
        optimizer.zero_grad()
        
        output = model1(X).squeeze()
        # print(output.shape)
        # print(X.shape)
        
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        correct, total = compute_accuracy(output, y)
        total_correct += correct
        total_samples += total
        
        true_positive, false_positive = compute_precision(output, y)
        total_true_positive += true_positive
        total_false_positive += false_positive
        
        false_negative = compute_recall(output, y)
        total_false_negative += false_negative
        

        
    scheduler.step()
    epoch_acc = total_correct / total_samples
    epoch_precision = total_true_positive/(total_true_positive+total_false_positive)
    epoch_recall = total_true_positive/(total_true_positive+total_false_negative)
    epoch_loss = total_loss/len(train_loader)
    
    train_loss.append(epoch_loss)
    train_acc.append(epoch_acc)
    train_precision.append(epoch_precision)
    train_recall.append(epoch_recall)
    print(f"Epoch {i+1} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.4f} | Precision: {epoch_precision:.4f} | Recall: {epoch_recall:.4f}")

    torch.save(
        model1.state_dict(),
        f"/blue/egn6933/apatil2/model_checkpoints/complex_model/model_epoch_{i+1}.pt"
    )
    

Epoch 1 | Loss: 0.4889 | Accuracy: 0.9089 | Precision: 0.2835 | Recall: 0.2728
Epoch 2 | Loss: 0.4576 | Accuracy: 0.9092 | Precision: 0.3217 | Recall: 0.3724
Epoch 3 | Loss: 0.4464 | Accuracy: 0.9089 | Precision: 0.3312 | Recall: 0.4091
Epoch 4 | Loss: 0.4370 | Accuracy: 0.9091 | Precision: 0.3401 | Recall: 0.4397
Epoch 5 | Loss: 0.4295 | Accuracy: 0.9092 | Precision: 0.3451 | Recall: 0.4590
Epoch 6 | Loss: 0.4225 | Accuracy: 0.9098 | Precision: 0.3513 | Recall: 0.4750
Epoch 7 | Loss: 0.4166 | Accuracy: 0.9103 | Precision: 0.3566 | Recall: 0.4905
Epoch 8 | Loss: 0.4108 | Accuracy: 0.9115 | Precision: 0.3645 | Recall: 0.5070
Epoch 9 | Loss: 0.4044 | Accuracy: 0.9123 | Precision: 0.3704 | Recall: 0.5192
Epoch 10 | Loss: 0.3982 | Accuracy: 0.9139 | Precision: 0.3792 | Recall: 0.5315


In [10]:
#Load the saved model weights
path = '/blue/egn6933/apatil2/model_checkpoints/model_epoch_5.pt'

checkpoint = torch.load(path)


In [11]:
#load the parameters into the MLP instance
model.load_state_dict(checkpoint)

<All keys matched successfully>

In [17]:
model.eval()

# predictions = []
# true_labels = []

test_samples = 0
correct_samples = 0
with torch.no_grad():

    for X, y in test_loader:
        # print(X.shape, y.shape)
        X = X.squeeze().to(device)
        y = y.squeeze().float().to(device)
        
        outputs = model(X).squeeze()
        # print(outputs.shape)
        # break
        test_loss = criterion(outputs, y).item()
        
        correct, total = compute_accuracy(outputs, y)
        correct_samples+=correct
        test_samples+=total
    
    print(f'The test accuracy is: {correct_samples/test_samples}')
        
    # print(X.shape, y.shape)

The test accuracy is: 0.9395266468903437
